## 🎯 Learning Objectives
* Design and implement a basic AI agent with perception, reasoning, and action capabilities.
* Integrate external 'tools' (simulated search and LLM) into an agent's workflow.
* Implement a simple memory mechanism for an agent to retain information.
* Orchestrate an agent's loop to perform a multi-step task like research and summarization.


## Exercise: Build a Personal Research Assistant Agent

Welcome to your first agent-building exercise! In previous lessons, we've explored the core components of AI agents: Perception, Reasoning, Action, Tools, and Memory. Now, it's time to put that knowledge into practice.

Your task is to build a simple **Personal Research Assistant Agent**. This agent should be able to take a research topic, find relevant information using a simulated search tool, summarize its findings, and then answer specific questions based on the gathered information.

### Task Description

Implement a Python class `ResearchAgent` that encapsulates the agent's logic. This agent will perform the following steps:

1.  **Perceive**: Understand the research topic and initial questions.
2.  **Act (Search)**: Use a provided `SearchTool` to find information related to the topic.
3.  **Memory**: Store the raw search results.
4.  **Reason & Act (Summarize)**: Use a provided `LLM` (Large Language Model) tool to summarize the gathered information.
5.  **Reason & Act (Answer Questions)**: Use the `LLM` tool again to answer specific questions based on the summarized information.
6.  **Output**: Present the summary and answers in a structured format.

### Requirements

*   Your `ResearchAgent` class must have an `__init__` method that accepts an `llm` instance and a `search_tool` instance.
*   It should have a `memory` attribute (e.g., a list or dictionary) to store perceived information.
*   Implement a `run(topic, questions)` method that orchestrates the entire research process.
*   Utilize the provided `MockSearchTool` and `MockLLM` classes (defined in the setup cell) to simulate external interactions.
*   Ensure your agent's output is clear and easy to understand.
*   Focus on modularity and clear separation of concerns within your agent's methods.

### Evaluation Criteria

*   **Correctness**: Does the agent successfully perform the research, summarization, and Q&A tasks using the mock tools?
*   **Agentic Principles**: Does the implementation clearly demonstrate perception, reasoning, action, and memory?
*   **Code Quality**: Is the code well-structured, readable, and commented?
*   **Completeness**: Does the agent handle the full workflow as described?
*   **Output Clarity**: Is the final output (summary and answers) well-formatted and informative?


In [ ]:
import json
from collections import defaultdict

# --- Mock Tools (Simulating external services) ---

class MockSearchTool:
    """A mock search tool that returns predefined results for specific queries."""
    def __init__(self):
        self.knowledge_base = {
            "quantum computing cybersecurity 2030": [
                "Quantum computing poses a significant threat to current asymmetric encryption (e.g., RSA, ECC) by 2030 due to Shor's algorithm.",
                "Post-quantum cryptography (PQC) is being developed to create new encryption standards resistant to quantum attacks.",
                "Governments and industries are investing heavily in quantum-resistant algorithms and quantum key distribution (QKD).",
                "The transition to PQC is complex and requires significant infrastructure upgrades and standardization efforts."
            ],
            "AI agent frameworks 2026": [
                "By 2026, advanced AI agent frameworks like AgenticFlow and AutogenX offer robust orchestration for multi-agent systems.",
                "These frameworks emphasize modularity, tool integration, and sophisticated memory management.",
                "They support dynamic tool selection and self-correction mechanisms for complex tasks.",
                "The adoption of standardized agent communication protocols is increasing."
            ],
            "impact of AI on education": [
                "AI is personalizing learning paths, offering adaptive content and real-time feedback.",
                "AI tutors and intelligent learning platforms are becoming more common, assisting both students and educators.",
                "Ethical concerns around data privacy, algorithmic bias, and the role of human educators are key discussion points.",
                "AI tools are automating administrative tasks, freeing up educators to focus on teaching."
            ]
        }

    def search(self, query: str) -> list[str]:
        """Simulates a web search for the given query."""
        print(f"[MockSearchTool] Searching for: '{query}'...")
        # Simple matching: check if query (case-insensitive) is a substring of any key
        for key, results in self.knowledge_base.items():
            if query.lower() in key.lower():
                return results
        return [f"No specific results found for '{query}'. General information might be available."]

class MockLLM:
    """A mock Large Language Model that simulates summarization and Q&A."""
    def __init__(self):
        pass

    def generate(self, prompt: str) -> str:
        """Simulates an LLM's response based on the prompt content."""
        print(f"[MockLLM] Generating response for prompt (first 50 chars): '{prompt[:50]}...' ")
        if "summarize the following information" in prompt.lower():
            # Extract content to summarize (very basic extraction for mock)
            content_start = prompt.lower().find("information:") + len("information:")
            content = prompt[content_start:].strip()
            if content:
                return f"Based on the provided information, here's a concise summary: {content[:150]}... (This is a mock summary, actual LLM would be more sophisticated)."
            else:
                return "(Mock Summary) No content provided for summarization."
        elif "answer the following questions" in prompt.lower():
            # Extract questions and context (very basic extraction)
            context_start = prompt.lower().find("context:") + len("context:")
            questions_start = prompt.lower().find("questions:") + len("questions:")
            
            context = prompt[context_start:questions_start-len("questions:")].strip()
            questions_str = prompt[questions_start:].strip()
            
            if not context or not questions_str:
                return "(Mock Q&A) Insufficient context or questions provided."

            # Simulate answering by just acknowledging the questions and context
            return f"(Mock Answer) Based on the context: '{context[:100]}...', I can answer your questions: '{questions_str[:100]}...' (Actual LLM would provide detailed answers)."
        else:
            return "(Mock LLM) I'm not sure how to respond to that prompt. Please use specific instructions for summarization or Q&A."

# Initialize the mock tools
mock_search_tool = MockSearchTool()
mock_llm = MockLLM()

print("Mock tools initialized successfully!")


### Your Turn: Implement the `ResearchAgent`

Now it's your turn to implement the `ResearchAgent` class. Use the `mock_search_tool` and `mock_llm` instances provided above. Remember to structure your agent with clear methods for perception, reasoning, and action, and to utilize its internal memory.

Think about:
*   How will the agent store the raw search results?
*   What prompts will you send to the `MockLLM` for summarization and Q&A?
*   How will you combine the perceived information to form a coherent output?

Good luck!


In [ ]:
class ResearchAgent:
    """A simple AI agent designed to research a topic, summarize, and answer questions."""
    def __init__(self, llm: MockLLM, search_tool: MockSearchTool):
        self.llm = llm
        self.search_tool = search_tool
        self.memory = {
            "raw_search_results": [],
            "summary": "",
            "answered_questions": {}
        } # Agent's internal memory to store findings
        print("[ResearchAgent] Initialized with LLM and SearchTool.")

    def _perceive(self, topic: str) -> str:
        """Perceives the initial research topic and uses the search tool to gather information."""
        print(f"[ResearchAgent] Perceiving topic: '{topic}'")
        # Action: Use the search tool
        results = self.search_tool.search(topic)
        self.memory["raw_search_results"] = results
        # Combine results into a single string for easier LLM processing
        perceived_info = "\n".join(results)
        print(f"[ResearchAgent] Perceived {len(results)} pieces of information.")
        return perceived_info

    def _reason_and_act_summarize(self, information: str) -> str:
        """Reasons that a summary is needed and acts by calling the LLM to summarize."""
        print("[ResearchAgent] Reasoning: Need to summarize gathered information.")
        prompt = f"Summarize the following information concisely:\n\nInformation:\n{information}"
        # Action: Call LLM for summarization
        summary = self.llm.generate(prompt)
        self.memory["summary"] = summary
        print("[ResearchAgent] Action: Summarized information.")
        return summary

    def _reason_and_act_answer_questions(self, context: str, questions: list[str]) -> dict:
        """Reasons that questions need answering and acts by calling the LLM."""
        print("[ResearchAgent] Reasoning: Need to answer specific questions based on context.")
        questions_str = "\n".join([f"- {q}" for q in questions])
        prompt = f"Based on the following context, answer the following questions:\n\nContext:\n{context}\n\nQuestions:\n{questions_str}"
        # Action: Call LLM for Q&A
        answers_raw = self.llm.generate(prompt)
        
        # In a real scenario, we'd parse the LLM's structured answer.
        # For this mock, we'll just store the raw mock answer.
        self.memory["answered_questions"] = {"raw_llm_response": answers_raw}
        print("[ResearchAgent] Action: Attempted to answer questions.")
        return self.memory["answered_questions"]

    def run(self, topic: str, questions: list[str]) -> dict:
        """Orchestrates the agent's perception-reasoning-action loop for research."""
        print(f"\n--- Research Agent Starting for Topic: '{topic}' ---")
        
        # 1. Perceive & Initial Act (Search)
        perceived_info = self._perceive(topic)
        
        # 2. Reason & Act (Summarize)
        summary = self._reason_and_act_summarize(perceived_info)
        
        # 3. Reason & Act (Answer Questions)
        # Use the summary as context for answering questions for better focus
        answered_q = self._reason_and_act_answer_questions(summary, questions)
        
        print("\n--- Research Agent Finished ---")
        return {
            "topic": topic,
            "raw_search_results": self.memory["raw_search_results"],
            "summary": self.memory["summary"],
            "answers": self.memory["answered_questions"]
        }

# --- Demonstrate the ResearchAgent --- 

# Instantiate the agent with our mock tools
research_agent = ResearchAgent(llm=mock_llm, search_tool=mock_search_tool)

# Define a research topic and specific questions
research_topic = "The impact of quantum computing on cybersecurity by 2030"
research_questions = [
    "What are the main threats quantum computing poses to current encryption methods?",
    "What countermeasures are being developed to address these threats?"
]

# Run the agent
research_report = research_agent.run(research_topic, research_questions)

# Print the final report
print("\n### Final Research Report ###")
print(f"Topic: {research_report['topic']}")
print("\n--- Raw Search Results ---")
for i, res in enumerate(research_report['raw_search_results']):
    print(f"{i+1}. {res}")

print("\n--- Summary ---")
print(research_report['summary'])

print("\n--- Answers to Questions ---")
# In a real agent, this would be parsed into individual Q&A pairs
print(research_report['answers']['raw_llm_response'])

print("\n--- Another Example: AI Agent Frameworks ---")
research_topic_2 = "AI agent frameworks 2026"
research_questions_2 = [
    "What are the key features of modern AI agent frameworks?",
    "How do they handle multi-agent systems?"
]

research_report_2 = research_agent.run(research_topic_2, research_questions_2)

print("\n### Final Research Report (Example 2) ###")
print(f"Topic: {research_report_2['topic']}")
print("\n--- Summary ---")
print(research_report_2['summary'])

print("\n--- Answers to Questions ---")
print(research_report_2['answers']['raw_llm_response'])
